# Генератор PL/SQL-обёртки (DBMS_LOB) под DAG

Один файл, всё внутри. Только стандартная библиотека — `pip install` не нужен.
Хвост `UTL_MD_UPSERT.upsert_FLOWTASKS` уже встроен по умолчанию.

**Как пользоваться:** `Run → Run All Cells`, затем работай в **Варианте А** (генерация прямо в ячейке — не требует localhost, работает на любом Jupyter).
Вариант Б (веб-форма) — опционально, только если Jupyter запущен локально на твоём компе.

Сначала обязательно выполни ячейку-движок ниже.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
gen_plsql_wrapper.py

Оборачивает произвольный SQL-скрипт в PL/SQL-генератор формата
    DECLARE ... DBMS_LOB.CREATETEMPORARY(v_SQL, TRUE);
            DBMS_LOB.APPEND(v_SQL, q'~ ... ~'); ...
    END;
который регистрирует текст SQL как шаг под DAG (Airflow).

Что делает за тебя:
  * режет SQL на куски строго < лимита VARCHAR2 (32767 байт), по границам строк;
  * считает длину в БАЙТАХ (UTF-8) — важно, т.к. кириллица в комментариях 2 байта;
  * сам подбирает q-разделитель (~ { [ ( < ...), если в куске встречается
    последовательность, которая закрыла бы литерал (напр. ~');
  * подставляет v_fd / v_wf (даг) / v_ts / v_patch_code (номер задачи);
  * по желанию добавляет PRE-SQL (truncate) и /*коммент-шага*/;
  * хвост-вызов фреймворка берёт из файла (--tail) и вставляет как есть.

Пример:
    python gen_plsql_wrapper.py \
        --sql        body.sql \
        --jira       PROJ-1234 \
        --dag        MY_DAG_NAME \
        --ts         my_load_step \
        --out        out.sql
        [--tail      tail_call.sql]
        [--target-table STAGING.MY_TARGET_TABLE]
        [--step-comment "my_mapping_step"]
"""

import argparse
import sys

# Лимит литерала VARCHAR2 в PL/SQL = 32767 байт. Берём запас.
MAX_BYTES_DEFAULT = 30000

# Кандидаты q-разделителей: (открывающий, закрывающий).
# Литерал q'<O> ... <C>' завершается последовательностью <C>'.
DELIMITERS = [
    ("~", "~"), ("{", "}"), ("[", "]"), ("(", ")"),
    ("<", ">"), ("!", "!"), ("|", "|"), ("#", "#"),
]


def byte_len(s: str, encoding: str = "utf-8") -> int:
    return len(s.encode(encoding))


def split_bytes(s: str, max_bytes: int) -> list:
    """Режет ОЧЕНЬ длинную строку (длиннее лимита) по границам символов."""
    out, cur, cur_b = [], [], 0
    for ch in s:
        cb = byte_len(ch)
        if cur and cur_b + cb > max_bytes:
            out.append("".join(cur))
            cur, cur_b = [ch], cb
        else:
            cur.append(ch)
            cur_b += cb
    if cur:
        out.append("".join(cur))
    return out


def to_units(sql: str, max_bytes: int) -> list:
    """SQL -> список 'юнитов' (строк), каждый гарантированно <= лимита."""
    units = []
    for line in sql.splitlines(keepends=True):
        if byte_len(line) <= max_bytes:
            units.append(line)
        else:
            units.extend(split_bytes(line, max_bytes))
    return units


def pack(units: list, max_bytes: int) -> list:
    """Жадно упаковывает юниты в куски <= лимита, не разрывая строки."""
    chunks, cur, cur_b = [], [], 0
    for u in units:
        ub = byte_len(u)
        if cur and cur_b + ub > max_bytes:
            chunks.append("".join(cur))
            cur, cur_b = [u], ub
        else:
            cur.append(u)
            cur_b += ub
    if cur:
        chunks.append("".join(cur))
    return chunks


def choose_delimiter(chunk: str):
    """Возвращает (open, close), для которого закрывающая последовательность
    <close>' НЕ встречается в куске. Иначе литерал закрылся бы раньше времени."""
    for o, c in DELIMITERS:
        if (c + "'") not in chunk:
            return o, c
    raise ValueError(
        "Не нашёл безопасный q-разделитель: кусок содержит все варианты "
        "закрывающих последовательностей. Уменьши --max-bytes или поправь SQL."
    )


def build_append(chunk: str) -> str:
    o, c = choose_delimiter(chunk)
    body = chunk.rstrip("\n")
    return f"  DBMS_LOB.APPEND(v_SQL, q'{o}\n{body}\n{c}');"


def sql_str_literal(val: str) -> str:
    """Безопасный строковый литерал для VARCHAR2 (удвоение одинарных кавычек)."""
    return "'" + val.replace("'", "''") + "'"


DEFAULT_TAIL = """  A := UTL_MD_UPSERT.upsert_FLOWTASKS
       (
           p_folder_name   => v_fd,
           p_workflow_name => v_wf,
           p_task_name     => v_ts,
           p_sql_text      => v_SQL,
           p_patch_code    => v_patch_code
       );"""


def generate(sql_text, jira, dag, v_ts, v_fd="AIRFLOW",
             tail=None, target_table=None, step_comment=None,
             max_bytes=MAX_BYTES_DEFAULT):
    warnings = []
    if len(jira) > 25:
        warnings.append(f"v_patch_code '{jira}' длиннее 25 символов (varchar2(25)).")
    if len(dag) > 255:
        warnings.append(f"v_wf (даг) длиннее 255 символов.")
    if len(v_ts) > 255:
        warnings.append(f"v_ts длиннее 255 символов.")

    # Префикс: коммент-шага и PRE-SQL truncate (всё попадёт в первый кусок).
    prefix = ""
    if step_comment:
        prefix += f"/*{step_comment}*/\n"
    if target_table:
        prefix += (f"--PRE SQL for table {target_table}\n"
                   f"truncate table {target_table}; --$$P_SRC_TABLE_NAME\n\n")
    body_sql = prefix + sql_text

    chunks = pack(to_units(body_sql, max_bytes), max_bytes)
    appends = "\n".join(build_append(c) for c in chunks)
    tail_block = tail.rstrip("\n") if tail else DEFAULT_TAIL

    plsql = f"""DECLARE
  v_fd         varchar2(255) := {sql_str_literal(v_fd)};
  v_wf         varchar2(255) := {sql_str_literal(dag)};
  v_ts         varchar2(255) := {sql_str_literal(v_ts)};
  v_patch_code varchar2(25)  := {sql_str_literal(jira)};
  A            varchar2(255);
  v_SQL        clob;
BEGIN
  DBMS_LOB.CREATETEMPORARY(v_SQL, TRUE);

{appends}

{tail_block}
END;
/
"""

    stats = {
        "chunks": len(chunks),
        "max_chunk_bytes": max(byte_len(c) for c in chunks) if chunks else 0,
        "total_bytes": byte_len(body_sql),
        "warnings": warnings,
    }
    return plsql, stats


def main(argv=None):
    p = argparse.ArgumentParser(
        description="Оборачивает SQL в PL/SQL-генератор (DBMS_LOB) для DAG-загрузки.")
    p.add_argument("--sql", required=True, help="путь к файлу с телом SQL")
    p.add_argument("--jira", required=True, help="номер задачи -> v_patch_code")
    p.add_argument("--dag", required=True, help="имя дага -> v_wf")
    p.add_argument("--ts", required=True, help="имя шага -> v_ts")
    p.add_argument("--out", help="куда писать (по умолчанию <ts>_dag.sql)")
    p.add_argument("--fd", default="AIRFLOW", help="v_fd (по умолчанию AIRFLOW)")
    p.add_argument("--tail", help="файл с хвостом-вызовом фреймворка (вставится как есть)")
    p.add_argument("--target-table", help="если задано -> добавит PRE-SQL truncate")
    p.add_argument("--step-comment", help="если задано -> добавит /*коммент*/ в начало")
    p.add_argument("--max-bytes", type=int, default=MAX_BYTES_DEFAULT,
                   help=f"лимит байт на кусок (по умолчанию {MAX_BYTES_DEFAULT})")
    args = p.parse_args(argv)

    with open(args.sql, encoding="utf-8") as f:
        sql_text = f.read()
    tail = None
    if args.tail:
        with open(args.tail, encoding="utf-8") as f:
            tail = f.read()

    plsql, stats = generate(
        sql_text=sql_text, jira=args.jira, dag=args.dag, v_ts=args.ts,
        v_fd=args.fd, tail=tail, target_table=args.target_table,
        step_comment=args.step_comment, max_bytes=args.max_bytes)

    out_path = args.out or f"{args.ts}_dag.sql"
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(plsql)

    print(f"OK -> {out_path}")
    print(f"  кусков APPEND: {stats['chunks']}")
    print(f"  макс. кусок:   {stats['max_chunk_bytes']} байт (лимит {args.max_bytes})")
    print(f"  всего SQL:     {stats['total_bytes']} байт")
    if not tail:
        print("  ВНИМАНИЕ: хвост не задан (--tail) — в файле стоит TODO-заглушка.")
    for w in stats["warnings"]:
        print(f"  ВНИМАНИЕ: {w}")
    return 0

print('Движок загружен (хвост upsert_FLOWTASKS уже встроен по умолчанию).')


## Вариант А — генерация в ячейке (рекомендуется)

Не зависит от localhost/IP/портов. Вставь GP-скрипт в `sql_gp`, заполни параметры, выполни. Результат печатается и сохраняется в `<v_ts>_dag.sql` со ссылкой на скачивание.

In [ ]:
# === Вариант А (ОСНОВНОЙ): генерация прямо в ячейке ===
# Вставь свой GP-скрипт между тройными кавычками, заполни параметры, выполни ячейку.

sql_gp = '''
ВСТАВЬ СЮДА СВОЙ GP-СКРИПТ ЦЕЛИКОМ
'''

jira         = "PROJECT-123"          # -> v_patch_code
dag          = "MY_DAG_NAME"          # -> v_wf
v_ts         = "my_load_step"         # -> v_ts
target_table = None                   # или "STAGING.MY_TABLE" -> добавит truncate
step_comment = None                   # или "имя_шага" -> добавит /*коммент*/

plsql, stats = generate(
    sql_text=sql_gp, jira=jira, dag=dag, v_ts=v_ts,
    target_table=target_table, step_comment=step_comment,
    # хвост upsert_FLOWTASKS встроен по умолчанию; tail=... только если нужен другой
)
print(stats)

out_file = v_ts + "_dag.sql"
with open(out_file, "w", encoding="utf-8") as f:
    f.write(plsql)

from IPython.display import FileLink, display
print("--- начало PL/SQL (первые 1500 символов) ---")
print(plsql[:1500])
print("--- полный текст сохранён в", out_file, "---")
display(FileLink(out_file))


## Вариант Б — веб-форма на localhost (опционально)

Сработает, только если Jupyter запущен на твоём компе. Если `localhost:8000` не открывается — значит Jupyter на сервере, оставайся на Варианте А.

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
serve.py — локальный веб-сервис для генерации PL/SQL-обёртки.

Запуск:
    python serve.py
Откроется браузер на http://localhost:8000 с формой: вставляешь SQL,
заполняешь поля -> получаешь готовый скрипт (копировать / скачать).

Без зависимостей (только стандартная библиотека). Работает локально,
слушает 127.0.0.1 — наружу в сеть не выставляется.

Требует рядом файл gen_plsql_wrapper.py (вся логика генерации — там).
"""

import argparse
import html
import json
import os
import sys
import threading
import webbrowser
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs

# generate() и MAX_BYTES_DEFAULT берутся из ячейки-движка выше


FORM_HTML = """<!doctype html>
<html lang="ru"><head><meta charset="utf-8">
<title>PL/SQL wrapper</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
  :root{--bg:#0f1115;--card:#171a21;--ink:#e6e8ec;--mut:#9aa3af;--line:#2a2f3a;--acc:#3b82f6;}
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 system-ui,Segoe UI,Roboto,sans-serif}
  .wrap{max-width:880px;margin:0 auto;padding:28px 20px 60px}
  h1{font-size:20px;margin:0 0 4px}
  .sub{color:var(--mut);margin:0 0 22px;font-size:13px}
  .card{background:var(--card);border:1px solid var(--line);border-radius:12px;padding:20px}
  label{display:block;font-size:13px;color:var(--mut);margin:14px 0 6px}
  input,textarea{width:100%;background:#0d0f14;color:var(--ink);border:1px solid var(--line);
    border-radius:8px;padding:10px 12px;font-family:ui-monospace,Consolas,monospace;font-size:13px}
  textarea{resize:vertical}
  .row{display:flex;gap:14px}.row>div{flex:1}
  .opt{color:var(--mut);font-weight:400}
  button{margin-top:20px;background:var(--acc);color:#fff;border:0;border-radius:8px;
    padding:11px 20px;font-size:14px;font-weight:600;cursor:pointer}
  button:hover{filter:brightness(1.08)}
  .hint{font-size:12px;color:var(--mut);margin-top:6px}
</style></head><body><div class="wrap">
  <h1>PL/SQL wrapper generator</h1>
  <p class="sub">Вставь тело SQL и параметры — получишь готовую DBMS_LOB-обёртку под DAG.</p>
  <form method="post" action="/generate" id="f"><div class="card">
    <label>SQL-скрипт *</label>
    <textarea name="sql" id="sql" rows="14" required placeholder="insert into ... with ... select ..."></textarea>
    <div class="row">
      <div><label>Номер задачи (v_patch_code) *</label><input name="jira" id="jira" required placeholder="PROJ-1234"></div>
      <div><label>DAG (v_wf) *</label><input name="dag" id="dag" required placeholder="MY_DAG_NAME"></div>
    </div>
    <div class="row">
      <div><label>v_ts (имя шага) *</label><input name="ts" id="ts" required placeholder="my_load_step"></div>
      <div><label>v_fd</label><input name="fd" id="fd" value="AIRFLOW"></div>
    </div>
    <div class="row">
      <div><label>Целевая таблица <span class="opt">(необяз. — добавит truncate)</span></label><input name="target_table" id="target_table" placeholder="STAGING.MY_TARGET_TABLE"></div>
      <div><label>Коммент-шага <span class="opt">(необяз.)</span></label><input name="step_comment" id="step_comment"></div>
    </div>
    <label>Хвост — вызов фреймворка <span class="opt">(необяз., вставится как есть; запомнится)</span></label>
    <textarea name="tail" id="tail" rows="4" placeholder="-- вызов процедуры, которая регистрирует v_SQL под DAG"></textarea>
    <label>Лимит байт на кусок</label><input name="max_bytes" id="max_bytes" value="__MAXB__">
    <div class="hint">VARCHAR2-литерал максимум 32767 байт; режется по строкам, с запасом.</div>
    <button type="submit">Сгенерировать</button>
  </div></form>
</div>
<script>
  // запоминаем «постоянные» поля между запусками (SQL не запоминаем — он меняется)
  var KEEP=["jira","dag","ts","fd","target_table","step_comment","tail","max_bytes"];
  KEEP.forEach(function(k){var v=localStorage.getItem("plsql_"+k);
    if(v!==null){var el=document.getElementById(k); if(el) el.value=v;}});
  document.getElementById("f").addEventListener("submit",function(){
    KEEP.forEach(function(k){var el=document.getElementById(k);
      if(el) localStorage.setItem("plsql_"+k, el.value);});
  });
</script>
</body></html>"""


RESULT_HTML = """<!doctype html>
<html lang="ru"><head><meta charset="utf-8"><title>Результат</title>
<meta name="viewport" content="width=device-width, initial-scale=1">
<style>
  :root{--bg:#0f1115;--card:#171a21;--ink:#e6e8ec;--mut:#9aa3af;--line:#2a2f3a;--acc:#3b82f6;--ok:#22c55e;--warn:#f59e0b;}
  *{box-sizing:border-box}
  body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.5 system-ui,Segoe UI,Roboto,sans-serif}
  .wrap{max-width:980px;margin:0 auto;padding:28px 20px 60px}
  h1{font-size:19px;margin:0 0 14px}
  a.back{color:var(--mut);text-decoration:none;font-size:13px}
  .stats{display:flex;gap:18px;flex-wrap:wrap;margin:0 0 14px;font-size:13px;color:var(--mut)}
  .stats b{color:var(--ink)}
  .warn{color:var(--warn)}
  textarea{width:100%;height:60vh;background:#0d0f14;color:var(--ink);border:1px solid var(--line);
    border-radius:10px;padding:14px;font-family:ui-monospace,Consolas,monospace;font-size:13px;white-space:pre}
  .bar{display:flex;gap:10px;margin:14px 0}
  button{background:var(--acc);color:#fff;border:0;border-radius:8px;padding:10px 18px;font-size:14px;font-weight:600;cursor:pointer}
  button.ghost{background:#222733}
</style></head><body><div class="wrap">
  <a class="back" href="/">← назад к форме</a>
  <h1>Готово</h1>
  <div class="stats">
    <span>кусков APPEND: <b>__CHUNKS__</b></span>
    <span>макс. кусок: <b>__MAXBYTES__</b> байт</span>
    <span>всего SQL: <b>__TOTAL__</b> байт</span>
    __WARN__
  </div>
  <div class="bar">
    <button onclick="copy()">Скопировать</button>
    <button class="ghost" onclick="download()">Скачать .sql</button>
  </div>
  <textarea id="out" readonly>__PLSQL__</textarea>
</div>
<script>
  function copy(){navigator.clipboard.writeText(document.getElementById("out").value)
    .then(function(){alert("Скопировано");});}
  function download(){
    var blob=new Blob([document.getElementById("out").value],{type:"text/plain"});
    var a=document.createElement("a");a.href=URL.createObjectURL(blob);
    a.download=__FILENAME__;document.body.appendChild(a);a.click();a.remove();}
</script>
</body></html>"""


def render_result(plsql, stats, filename):
    warn = ""
    if stats["warnings"]:
        warn = '<span class="warn">⚠ ' + " · ".join(html.escape(w) for w in stats["warnings"]) + "</span>"
    return (RESULT_HTML
            .replace("__CHUNKS__", str(stats["chunks"]))
            .replace("__MAXBYTES__", str(stats["max_chunk_bytes"]))
            .replace("__TOTAL__", str(stats["total_bytes"]))
            .replace("__WARN__", warn)
            .replace("__PLSQL__", html.escape(plsql))
            .replace("__FILENAME__", json.dumps(filename)))


class Handler(BaseHTTPRequestHandler):
    def _send(self, body, code=200):
        data = body.encode("utf-8")
        self.send_response(code)
        self.send_header("Content-Type", "text/html; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        if self.path in ("/", "/index.html"):
            self._send(FORM_HTML.replace("__MAXB__", str(MAX_BYTES_DEFAULT)))
        else:
            self._send("<h1>404</h1><a href='/'>на форму</a>", 404)

    def do_POST(self):
        if self.path != "/generate":
            self._send("<h1>404</h1>", 404)
            return
        length = int(self.headers.get("Content-Length", 0))
        raw = self.rfile.read(length).decode("utf-8")
        f = parse_qs(raw, keep_blank_values=True)

        def g(k, default=""):
            return f.get(k, [default])[0]

        try:
            sql = g("sql")
            jira, dag, ts = g("jira"), g("dag"), g("ts")
            if not (sql.strip() and jira and dag and ts):
                raise ValueError("Заполни обязательные поля: SQL, jira, DAG, v_ts.")
            try:
                max_bytes = int(g("max_bytes", str(MAX_BYTES_DEFAULT)))
            except ValueError:
                max_bytes = MAX_BYTES_DEFAULT
            plsql, stats = generate(
                sql_text=sql, jira=jira, dag=dag, v_ts=ts,
                v_fd=g("fd", "AIRFLOW") or "AIRFLOW",
                tail=g("tail") or None,
                target_table=g("target_table") or None,
                step_comment=g("step_comment") or None,
                max_bytes=max_bytes)
            self._send(render_result(plsql, stats, f"{ts}_dag.sql"))
        except Exception as e:
            self._send(f"<div style='font:15px system-ui;color:#eee;background:#0f1115;"
                       f"padding:24px'><h1>Ошибка</h1><p>{html.escape(str(e))}</p>"
                       f"<a style='color:#9aa3af' href='/'>← назад</a></div>", 400)

    def log_message(self, *a):  # тише в консоли
        pass

# --- запуск внутри Jupyter: фоновый поток + ссылка (работает только если Jupyter локальный) ---
_httpd = globals().get("_httpd")
if _httpd is not None:
    try:
        _httpd.shutdown()
    except Exception:
        pass

PORT = 8000
_httpd = ThreadingHTTPServer(("127.0.0.1", PORT), Handler)
threading.Thread(target=_httpd.serve_forever, daemon=True).start()

from IPython.display import display, HTML
display(HTML(f'<a href="http://localhost:{PORT}" target="_blank">http://localhost:{PORT}</a>'))
print("Если ссылка не открывается — Jupyter на удалённом сервере. Используй Вариант А (ячейка) выше.")
